# Grid de pesos do Composable CFG — s_id x s_clip x s_attr (ou s_id x s_attr)

Gera via **DDIM** a partir de uma foto (sem editar atributos — o vetor usado é o
da própria foto), testando todas as combinações das escalas de guidance.

O notebook detecta o `encoder` salvo no checkpoint:
- `clip_arcface_split` (ramo CLIP separado do ArcFace) → grid **3-D**
  (`s_id x s_clip x s_attr`), 1000 combinações.
- qualquer outro (`clip_arcface`, `arcface_only`) → grid **2-D**
  (`s_id x s_attr`), 100 combinações, igual ao `generate_from_photo.py`
  (não existe um `s_clip` independente nesses checkpoints).

Para isso ser viável, o sampler abaixo é **batched**: os tokens de condição são
os mesmos para todas as combinações (mesma foto, mesmos atributos), só as
escalas mudam — então várias combinações rodam em paralelo no mesmo forward
do UNet, com as escalas como tensores `[N,1,1,1]`.

Todas as combinações partem do **mesmo z_T** (mesma seed) e o DDIM é
determinístico (eta=0) — a única diferença entre as imagens são os pesos.

Saída em `results/grid_weights/`:
- `full_grid/` — as imagens individuais;
- grid 3-D: `grid_sattr{a}_seed{s}.png`, um por `s_attr` (linhas=`s_id`, colunas=`s_clip`);
- grid 2-D: `grid_seed{s}.png` (linhas=`s_id`, colunas=`s_attr`).


In [ ]:
# ============ CONFIG ============
PHOTO           = "nova.jpeg"
CKPT            = "models/LDM_CFGComp_split_paired/ckpt_best.pt"   # 3 pesos -> checkpoint clip_arcface_split
VAE_CKPT        = "vae/vae_epoch_62.pt"
CLASSIFIER_CKPT = "models/attribute_classifier/ckpt_best.pt"        # None p/ usar ORIG_ATTRS
ORIG_ATTRS      = ["Young", "Male", "Black_Hair", "Mustache"]       # usado só se CLASSIFIER_CKPT=None

S_ID_VALUES   = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
S_CLIP_VALUES = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
S_ATTR_VALUES = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

CHUNK      = 50    # combinações geradas em paralelo por chunk (reduza se der OOM)
DDIM_STEPS = 50
SEED       = 0
DEVICE     = "cuda"
SAVE_DIR   = "results/grid_weights"


In [ ]:
# ============ PIPELINE + FOTO + ATRIBUTOS ============
import os
from types import SimpleNamespace

import torch
import matplotlib.pyplot as plt
from torchvision.utils import save_image
from tqdm.auto import tqdm

from utils.edit_common import (
    EditPipeline, prepare_photo, resolve_original_attrs, CELEBA_ATTRS,
)

os.makedirs(os.path.join(SAVE_DIR, "full_grid"), exist_ok=True)

ref_img = prepare_photo(PHOTO, DEVICE)          # [1,3,256,256] em [-1,1]
pipeline = EditPipeline(ckpt_path=CKPT, vae_ckpt=VAE_CKPT, device=DEVICE)
print(f"encoder: {pipeline.encoder_type}")

# "clip_arcface_split" tem ramo CLIP separado do ArcFace -> grid 3-D
# (s_id x s_clip x s_attr). Qualquer outro encoder ("clip_arcface",
# "arcface_only") não tem s_clip independente -> cai para o grid 2-D
# (s_id x s_attr), igual ao generate_from_photo.py.
IS_SPLIT = pipeline.encoder_type == "clip_arcface_split"
print("modo:", "3-D (s_id x s_clip x s_attr)" if IS_SPLIT
      else "2-D (s_id x s_attr) -- checkpoint sem ramo CLIP separado")

# atributos da própria foto, SEM edição
_args = SimpleNamespace(orig_attrs=None if CLASSIFIER_CKPT else ORIG_ATTRS,
                        classifier_ckpt=CLASSIFIER_CKPT)
attrs_vec = resolve_original_attrs(_args, PHOTO, DEVICE)
print("Atributos usados:", [CELEBA_ATTRS[i] for i in range(40) if attrs_vec[i] == 1.0])


In [ ]:
# ============ TOKENS (calculados uma vez) ============
with torch.no_grad():
    attr_tok = pipeline.attribute_embedder(attrs_vec.view(1, -1).to(DEVICE))
    if IS_SPLIT:
        clip_tok, id_tok = pipeline.image_encoder(ref_img=ref_img, return_separate=True)
    else:
        id_tok = pipeline.image_encoder(ref_img=ref_img)
        clip_tok = None

ref_vis = ((ref_img.squeeze(0).clamp(-1, 1) + 1) / 2).cpu()
plt.figure(figsize=(3, 3)); plt.imshow(ref_vis.permute(1, 2, 0)); plt.axis("off")
plt.title("entrada alinhada"); plt.show()


In [ ]:
# ============ SAMPLER DDIM BATCHED (escalas por amostra) ============
# Mesma matemática de utils/composable_cfg_sampling.sample_split_ddim /
# sample_composable_ddim, mas as escalas são tensores [N,1,1,1] — cada
# amostra do batch usa a sua própria combinação, compartilhando os forwards
# do UNet. Dispatcha entre a cadeia de 3 ramos (split) e a de 2 ramos
# (clip_arcface / arcface_only) conforme IS_SPLIT.

@torch.no_grad()
def sample_batch(combos, seed=0, ddim_steps=50, channels=4):
    """combos: lista de (s_id, s_clip, s_attr).
    Se IS_SPLIT=False, s_clip é ignorado. Retorna latentes [N,4,32,32]."""
    unet, diffusion = pipeline.unet, pipeline.diffusion
    N = len(combos)
    img_size = diffusion.img_size

    torch.manual_seed(seed)
    z_t = torch.randn(1, channels, img_size, img_size, device=DEVICE).repeat(N, 1, 1, 1)

    s_id   = torch.tensor([c[0] for c in combos], device=DEVICE, dtype=torch.float32).view(N, 1, 1, 1)
    s_attr = torch.tensor([c[2] for c in combos], device=DEVICE, dtype=torch.float32).view(N, 1, 1, 1)

    def rep(ctx):  # [1,C,T] -> [N,C,T]
        return ctx.repeat(N, 1, 1)

    if IS_SPLIT:
        s_clip = torch.tensor([c[1] for c in combos], device=DEVICE, dtype=torch.float32).view(N, 1, 1, 1)
        zeros_id, zeros_clip, zeros_attr = (
            torch.zeros_like(id_tok), torch.zeros_like(clip_tok), torch.zeros_like(attr_tok),
        )
        ctx_000 = rep(torch.cat([zeros_id, zeros_clip, zeros_attr], dim=2))
        ctx_i00 = rep(torch.cat([id_tok,   zeros_clip, zeros_attr], dim=2))
        ctx_ic0 = rep(torch.cat([id_tok,   clip_tok,   zeros_attr], dim=2))
        ctx_ica = rep(torch.cat([id_tok,   clip_tok,   attr_tok],   dim=2))
    else:
        zeros_id, zeros_attr = torch.zeros_like(id_tok), torch.zeros_like(attr_tok)
        ctx_uu = rep(torch.cat([zeros_id, zeros_attr], dim=2))
        ctx_iu = rep(torch.cat([id_tok,   zeros_attr], dim=2))
        ctx_ia = rep(torch.cat([id_tok,   attr_tok],   dim=2))

    step_size = max(diffusion.noise_steps // ddim_steps, 1)
    times = list(range(1, diffusion.noise_steps, step_size))[::-1]

    for idx, t in enumerate(times):
        t_vec = torch.full((N,), t, device=DEVICE, dtype=torch.long)

        if IS_SPLIT:
            eps_000 = unet(z_t, t_vec, context=ctx_000)
            eps_i00 = unet(z_t, t_vec, context=ctx_i00)
            eps_ic0 = unet(z_t, t_vec, context=ctx_ic0)
            eps_ica = unet(z_t, t_vec, context=ctx_ica)
            eps = (
                eps_000
                + s_id   * (eps_i00 - eps_000)
                + s_clip * (eps_ic0 - eps_i00)
                + s_attr * (eps_ica - eps_ic0)
            )
        else:
            eps_uu = unet(z_t, t_vec, context=ctx_uu)
            eps_iu = unet(z_t, t_vec, context=ctx_iu)
            eps_ia = unet(z_t, t_vec, context=ctx_ia)
            eps = eps_uu + s_id * (eps_iu - eps_uu) + s_attr * (eps_ia - eps_iu)

        alpha_bar_t = diffusion.alpha_hat[t]
        is_last = (idx == len(times) - 1)
        alpha_bar_next = (
            torch.tensor(1.0, device=DEVICE) if is_last
            else diffusion.alpha_hat[times[idx + 1]]
        )

        pred_x0 = (z_t - torch.sqrt(1.0 - alpha_bar_t) * eps) / torch.sqrt(alpha_bar_t)
        dir_xt = torch.sqrt(torch.clamp(1.0 - alpha_bar_next, min=0.0)) * eps
        z_t = torch.sqrt(alpha_bar_next) * pred_x0 + dir_xt

    return z_t


In [ ]:
# ============ RODA AS COMBINAÇÕES (em chunks) ============
# 3-D (split): s_attr externo, depois s_id, depois s_clip -- 1000 combinações
# (assim as 100 imagens de cada grid 10x10 ficam contíguas).
# 2-D (não-split): s_id x s_attr -- 100 combinações; s_clip fixo em 0.0
# (não usado pelo sampler nesse modo).
if IS_SPLIT:
    all_combos = [
        (float(si), float(sc), float(sa))
        for sa in S_ATTR_VALUES
        for si in S_ID_VALUES
        for sc in S_CLIP_VALUES
    ]
else:
    all_combos = [
        (float(si), 0.0, float(sa))
        for sa in S_ATTR_VALUES
        for si in S_ID_VALUES
    ]
print(f"{len(all_combos)} combinações, chunks de {CHUNK}")

images_u8 = {}   # combo -> uint8 [3,256,256]

for start in tqdm(range(0, len(all_combos), CHUNK), desc="chunks"):
    chunk = all_combos[start:start + CHUNK]
    latents = sample_batch(chunk, seed=SEED, ddim_steps=DDIM_STEPS)

    # decode do VAE em sub-lotes menores (ativações do decoder são grandes)
    for i in range(0, len(chunk), 10):
        with torch.no_grad():
            imgs = pipeline.decode(latents[i:i + 10])
        imgs = ((imgs.clamp(-1, 1) + 1) / 2 * 255).byte().cpu()
        for j, combo in enumerate(chunk[i:i + 10]):
            images_u8[combo] = imgs[j]
            si, sc, sa = (int(v) for v in combo)
            name = (f"sid{si}_sclip{sc}_sattr{sa}.png" if IS_SPLIT
                    else f"sid{si}_sattr{sa}.png")
            save_image(imgs[j].float() / 255, os.path.join(SAVE_DIR, "full_grid", name))

print(f"OK — {len(images_u8)} imagens em {SAVE_DIR}/full_grid/")


In [ ]:
# ============ MONTA O(S) GRID(S) ============
if IS_SPLIT:
    # um grid 10x10 por s_attr: linhas = s_id, colunas = s_clip
    for sa in S_ATTR_VALUES:
        tiles = [
            images_u8[(float(si), float(sc), float(sa))].float() / 255
            for si in S_ID_VALUES
            for sc in S_CLIP_VALUES
        ]
        out = os.path.join(SAVE_DIR, f"grid_sattr{int(sa)}_seed{SEED}.png")
        save_image(torch.stack(tiles), out, nrow=len(S_CLIP_VALUES))
        print(f"salvo: {out}")
else:
    # um único grid: linhas = s_id, colunas = s_attr
    tiles = [
        images_u8[(float(si), 0.0, float(sa))].float() / 255
        for si in S_ID_VALUES
        for sa in S_ATTR_VALUES
    ]
    out = os.path.join(SAVE_DIR, f"grid_seed{SEED}.png")
    save_image(torch.stack(tiles), out, nrow=len(S_ATTR_VALUES))
    print(f"salvo: {out}")


In [ ]:
# ============ VISUALIZAÇÃO RÁPIDA ============
if IS_SPLIT:
    SA_SHOW = 5   # qual s_attr mostrar inline
    n_id, n_clip = len(S_ID_VALUES), len(S_CLIP_VALUES)
    fig, axes = plt.subplots(n_id, n_clip, figsize=(1.6 * n_clip, 1.7 * n_id))
    for i, si in enumerate(S_ID_VALUES):
        for j, sc in enumerate(S_CLIP_VALUES):
            ax = axes[i][j]
            ax.imshow(images_u8[(float(si), float(sc), float(SA_SHOW))].permute(1, 2, 0).numpy())
            ax.axis("off")
            if i == 0: ax.set_title(f"s_clip={sc}", fontsize=8)
            if j == 0: ax.text(-0.1, 0.5, f"s_id={si}", fontsize=8, rotation=90,
                               va="center", ha="center", transform=ax.transAxes)
    fig.suptitle(f"s_id x s_clip   (s_attr={SA_SHOW}, seed={SEED})", fontsize=12)
else:
    n_id, n_attr = len(S_ID_VALUES), len(S_ATTR_VALUES)
    fig, axes = plt.subplots(n_id, n_attr, figsize=(1.6 * n_attr, 1.7 * n_id))
    for i, si in enumerate(S_ID_VALUES):
        for j, sa in enumerate(S_ATTR_VALUES):
            ax = axes[i][j]
            ax.imshow(images_u8[(float(si), 0.0, float(sa))].permute(1, 2, 0).numpy())
            ax.axis("off")
            if i == 0: ax.set_title(f"s_attr={sa}", fontsize=8)
            if j == 0: ax.text(-0.1, 0.5, f"s_id={si}", fontsize=8, rotation=90,
                               va="center", ha="center", transform=ax.transAxes)
    fig.suptitle(f"s_id x s_attr   (seed={SEED})", fontsize=12)
plt.tight_layout(); plt.show()


## Como ler

**Modo 3-D** (checkpoint `clip_arcface_split`):
- Dentro de um grid (`s_attr` fixo): descendo uma coluna, `s_id` cresce —
  a identidade deve travar no seu rosto; alto demais → saturação/artefatos.
  Indo para a direita numa linha, `s_clip` cresce — aparência global
  (expressão, iluminação, cabelo) puxa mais para a foto.
- Entre grids: `s_attr` cresce — os atributos ficam mais impostos e
  começam a competir com a identidade.

**Modo 2-D** (checkpoint `clip_arcface` / `arcface_only`):
- `grid_seed{s}.png`: descendo uma coluna, `s_id` cresce (identidade trava
  no rosto); indo para a direita numa linha, `s_attr` cresce (atributos mais
  impostos, competindo mais com a identidade).

Anote as 2–3 melhores células (`sid/(sclip/)sattr` está no nome do arquivo em
`full_grid/`) e use esses pesos em `generate_from_photo.py`.
